# 01 — SPARCS Data and Schema Audit

## Purpose

This notebook audits the structure, quality, and analytical usability of the
2023 New York State SPARCS de-identified inpatient discharge dataset.

## Main Method

Descriptive data-quality profiling using Python, pandas, and DuckDB.

No predictive model is trained in this notebook.

## Goals

1. Validate the source file and schema.
2. Confirm the analytical grain and dataset dimensions.
3. Measure missingness and cardinality.
4. inspect categorical values and suppressed fields.
5. Validate numeric parsing.
6. Examine LOS, charge, and estimated-cost distributions.
7. Identify LOS top-coding.
8. Assess apparent duplicate records.
9. Evaluate facility-key consistency.
10. Document implications for modeling and Power BI.

## Out of Scope

- Data cleaning or recoding
- Missing-value imputation
- Duplicate removal
- Feature engineering
- Model training
- Power BI ingestion
- Multi-year dataset combination

The analytical unit is one inpatient discharge. Because no unique discharge
identifier is expected, apparently identical rows cannot automatically be
treated as duplicate errors.

## 1. Imports

In [1]:
from pathlib import Path
import hashlib

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

## 2. Project Paths

In [2]:
def find_project_root(start_path):
    """Find the repository root using the existing project charter."""

    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        charter_path = candidate / "docs" / "project_charter.md"

        if charter_path.exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Expected docs/project_charter.md "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "sparcs_inpatient_2023.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Source file not found: {DATA_PATH}"

print("Project root detected successfully.")
print("Project folder:", PROJECT_ROOT.name)
print("Data Path detected successfully.")
print("Data File:", DATA_PATH.name)
print("Output Directory detected successfully.")
print("Output Directory:", OUTPUT_DIR.name)

Project root detected successfully.
Project folder: 04_hospital_operations_powerbi
Data Path detected successfully.
Data File: sparcs_inpatient_2023.csv
Output Directory detected successfully.
Output Directory: data_audit


### Interpretation

The project root was detected using `docs/project_charter.md` rather than a
hard-coded local path. The notebook can therefore be launched from either the
repository root or the `notebooks` directory.

## 3. File metadata and reproducibility

In [3]:
def calculate_sha256(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


file_metadata = pd.DataFrame(
    [
        {
            "file_name": DATA_PATH.name,
            "file_size_gb": round(
                DATA_PATH.stat().st_size / (1024 ** 3),
                3,
            ),
            "sha256": calculate_sha256(DATA_PATH),
        }
    ]
)

file_metadata

,file_name,file_size_gb,sha256
0,sparcs_inpatient_2023.csv,0.882,d69e4b9e47fd2992c323c858cde788578b085e2ba7e0eab7ee8d961db0df97cb


## 4. Load the file with DuckDB

In [4]:
con = duckdb.connect(database=":memory:")

raw_relation = con.read_csv(
    str(DATA_PATH),
    header=True,
    all_varchar=True,
    sample_size=100_000,
)

_ = raw_relation.create_view("sparcs_raw")

## 5. Dataset structure

In [5]:
row_count = con.sql("""
    SELECT COUNT(*) AS row_count
    FROM sparcs_raw
""").df()

schema = con.sql("""
    DESCRIBE sparcs_raw
""").df()

total_rows = int(row_count.loc[0, "row_count"])
columns = schema["column_name"].tolist()

assert total_rows > 0
assert len(columns) == schema.shape[0]

display(row_count)
display(schema)

print(f"Rows:    {total_rows:,}")
print(f"Columns: {len(columns)}")
print(columns)

,row_count
0,2125754


,column_name,column_type,null,key,default,extra
0,Hospital Service Area,VARCHAR,YES,None,None,None
1,Hospital County,VARCHAR,YES,None,None,None
2,Operating Certificate Number,VARCHAR,YES,None,None,None
3,Permanent Facility Id,VARCHAR,YES,None,None,None
4,Facility Name,VARCHAR,YES,None,None,None
5,Age Group,VARCHAR,YES,None,None,None
6,Zip Code - 3 digits,VARCHAR,YES,None,None,None
7,Gender,VARCHAR,YES,None,None,None
8,Race,VARCHAR,YES,None,None,None
9,Ethnicity,VARCHAR,YES,None,None,None


Rows:    2,125,754
Columns: 33
['Hospital Service Area', 'Hospital County', 'Operating Certificate Number', 'Permanent Facility Id', 'Facility Name', 'Age Group', 'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Length of Stay', 'Type of Admission', 'Patient Disposition', 'Discharge Year', 'CCSR Diagnosis Code', 'CCSR Diagnosis Description', 'CCSR Procedure Code', 'CCSR Procedure Description', 'APR DRG Code', 'APR DRG Description', 'APR MDC Code', 'APR MDC Description', 'APR Severity of Illness Code', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3', 'Birth Weight', 'Emergency Department Indicator', 'Total Charges', 'Total Costs']


### Interpretation

- The dataset contains **2,125,754 released inpatient discharge records** and
  **33 source fields**.
- The intended analytical grain is one de-identified inpatient discharge
  record per row. However, the absence of a unique discharge identifier means
  row-level uniqueness cannot be independently confirmed.
- The only available time field is `Discharge Year`, and every record belongs
  to 2023.
- A unique patient or discharge identifier is not available.
- Monthly analysis is not supported because the file contains no admission
  date, discharge date, discharge month, or equivalent within-year time field.

## 6. Required-field availability

In [6]:
required_fields = {
    "Facility identifier": "Permanent Facility Id",
    "Operating certificate": "Operating Certificate Number",
    "Facility name": "Facility Name",
    "Hospital service area": "Hospital Service Area",
    "Hospital county": "Hospital County",
    "Length of stay": "Length of Stay",
    "Admission type": "Type of Admission",
    "Patient disposition": "Patient Disposition",
    "Discharge year": "Discharge Year",
    "APR-DRG code": "APR DRG Code",
    "APR-DRG description": "APR DRG Description",
    "Severity code": "APR Severity of Illness Code",
    "Severity description": "APR Severity of Illness Description",
    "Mortality risk": "APR Risk of Mortality",
    "Medical/surgical classification": (
        "APR Medical Surgical Description"
    ),
    "Primary payer": "Payment Typology 1",
    "Emergency department indicator": (
        "Emergency Department Indicator"
    ),
    "Total charges": "Total Charges",
    "Estimated costs": "Total Costs",
    "CCSR diagnosis code": "CCSR Diagnosis Code",
    "CCSR procedure code": "CCSR Procedure Code",
}

field_availability = pd.DataFrame(
    [
        {
            "business_concept": concept,
            "expected_source_field": field,
            "available": field in columns,
        }
        for concept, field in required_fields.items()
    ]
)

field_availability

,business_concept,expected_source_field,available
0,Facility identifier,Permanent Facility Id,True
1,Operating certificate,Operating Certificate Number,True
2,Facility name,Facility Name,True
3,Hospital service area,Hospital Service Area,True
4,Hospital county,Hospital County,True
5,Length of stay,Length of Stay,True
6,Admission type,Type of Admission,True
7,Patient disposition,Patient Disposition,True
8,Discharge year,Discharge Year,True
9,APR-DRG code,APR DRG Code,True


In [7]:
missing_required_fields = field_availability.loc[
    ~field_availability["available"],
    "expected_source_field",
].tolist()

print("Missing expected fields:", missing_required_fields)

Missing expected fields: []


## 7. Missingness and cardinality

In [8]:
def quote_identifier(column_name):
    return '"' + column_name.replace('"', '""') + '"'


def quote_literal(value):
    return "'" + value.replace("'", "''") + "'"


profile_queries = []

for column in columns:
    column_sql = quote_identifier(column)

    profile_queries.append(
        f"""
        SELECT
            {quote_literal(column)} AS column_name,
            SUM(
                CASE
                    WHEN {column_sql} IS NULL
                      OR TRIM({column_sql}) = ''
                    THEN 1
                    ELSE 0
                END
            ) AS missing_n,
            COUNT(
                DISTINCT NULLIF(TRIM({column_sql}), '')
            ) AS unique_n
        FROM sparcs_raw
        """
    )

column_profile = con.sql(
    " UNION ALL ".join(profile_queries)
).df()

column_profile["missing_pct"] = (
    column_profile["missing_n"] / total_rows * 100
)

column_profile = column_profile.sort_values(
    ["missing_pct", "unique_n"],
    ascending=[False, False],
).reset_index(drop=True)

column_profile

,column_name,missing_n,unique_n,missing_pct
0,Birth Weight,1921751.0,81,90.403264
1,Payment Typology 3,1843136.0,9,86.705047
2,Payment Typology 2,1114046.0,9,52.407099
3,CCSR Procedure Code,611024.0,320,28.743872
4,CCSR Procedure Description,611024.0,320,28.743872
5,Zip Code - 3 digits,41883.0,50,1.970266
6,Permanent Facility Id,5333.0,207,0.250876
7,Operating Certificate Number,5333.0,160,0.250876
8,Hospital County,5333.0,57,0.250876
9,Hospital Service Area,5333.0,8,0.250876


### Interpretation

- The fields with the highest missingness are `Birth Weight` at **90.40%**,
  `Payment Typology 3` at **86.71%**, `Payment Typology 2` at **52.41%**,
  and the CCSR procedure code and description fields at **28.74%**.
- Missingness in birth weight, secondary and tertiary payer fields, and
  procedure fields is likely to be at least partly structural or
  not-applicable. This interpretation must be confirmed during data cleaning.
- Additional missingness requiring review occurs in `Zip Code - 3 digits`
  at **1.97%**, facility and geographic identifiers at approximately
  **0.25%**, and APR severity and mortality fields at **0.02%**.
- `Discharge Year` is constant because the audited file contains only 2023
  records. No other near-constant field was established by this audit.
- High-cardinality fields include `Total Charges`, `Total Costs`,
  CCSR diagnosis, APR-DRG, CCSR procedure, and facility. Continuous financial
  values should be modeled as measures rather than categorical Power BI
  dimensions.

Missing values have not been imputed or recoded.

## 8. Categorical-value audit

In [9]:
categorical_columns = [
    "Hospital Service Area",
    "Hospital County",
    "Operating Certificate Number",
    "Permanent Facility Id",
    "Facility Name",
    "Age Group",
    "Zip Code - 3 digits",
    "Gender",
    "Race",
    "Ethnicity",
    "Type of Admission",
    "Patient Disposition",
    "APR Severity of Illness Description",
    "APR Risk of Mortality",
    "APR Medical Surgical Description",
    "Payment Typology 1",
    "Payment Typology 2",
    "Payment Typology 3",
    "Emergency Department Indicator",
    "Discharge Year",
]

categorical_results = []

for column in categorical_columns:
    if column not in columns:
        continue

    column_sql = quote_identifier(column)

    result = con.sql(
        f"""
        SELECT
            {quote_literal(column)} AS column_name,
            COALESCE(
                NULLIF(TRIM({column_sql}), ''),
                '[MISSING]'
            ) AS category,
            COUNT(*) AS discharges,
            ROUND(
                COUNT(*) * 100.0 / {total_rows},
                4
            ) AS percentage
        FROM sparcs_raw
        GROUP BY 1, 2
        ORDER BY discharges DESC
        """
    ).df()

    categorical_results.append(result)

categorical_counts = pd.concat(
    categorical_results,
    ignore_index=True,
)

categorical_counts.groupby(
    "column_name",
    group_keys=False,
).head(20)

,column_name,category,discharges,percentage
0,Hospital Service Area,New York City,963525,45.3263
1,Hospital Service Area,Long Island,346694,16.3092
2,Hospital Service Area,Hudson Valley,237546,11.1747
3,Hospital Service Area,Western NY,138654,6.5226
4,Hospital Service Area,Finger Lakes,137867,6.4856
...,...,...,...,...
774,Payment Typology 3,Miscellaneous/Other,239,0.0112
775,Payment Typology 3,Department of Corrections,22,0.0010
776,Emergency Department Indicator,Y,1335889,62.8431
777,Emergency Department Indicator,N,789865,37.1569


In [10]:
categorical_counts.loc[
    categorical_counts["column_name"] == "Patient Disposition"
]

,column_name,category,discharges,percentage
715,Patient Disposition,Home or Self Care,1405879,66.1355
716,Patient Disposition,Home w/ Home Health Services,281903,13.2613
717,Patient Disposition,Skilled Nursing Home,200275,9.4214
718,Patient Disposition,Left Against Medical Advice,58445,2.7494
719,Patient Disposition,Expired,53845,2.5330
720,Patient Disposition,Short-term Hospital,36246,1.7051
721,Patient Disposition,Inpatient Rehabilitation Facility,35392,1.6649
722,Patient Disposition,Hospice - Home,11159,0.5249
723,Patient Disposition,Hospice - Medical Facility,9846,0.4632
724,Patient Disposition,Psychiatric Hospital or Unit of Hosp,9726,0.4575


## 9. Numeric-parsing validation

In [11]:
numeric_columns = [
    "Length of Stay",
    "Birth Weight",
    "Total Charges",
    "Total Costs",
    "APR DRG Code",
    "APR MDC Code",
    "APR Severity of Illness Code",
]


def numeric_expression(column):
    column_sql = quote_identifier(column)

    return f"""
        TRY_CAST(
            NULLIF(
                REPLACE(
                    REPLACE(TRIM({column_sql}), '$', ''),
                    ',',
                    ''
                ),
                ''
            )
            AS DOUBLE
        )
    """


numeric_results = []
nonnumeric_results = []

for column in numeric_columns:
    if column not in columns:
        continue

    column_sql = quote_identifier(column)
    expression = numeric_expression(column)

    numeric_results.append(
        con.sql(
            f"""
            SELECT
                {quote_literal(column)} AS column_name,
                COUNT(*) AS total_n,
                COUNT(NULLIF(TRIM({column_sql}), ''))
                    AS nonmissing_n,
                COUNT({expression}) AS numeric_n,
                COUNT(NULLIF(TRIM({column_sql}), ''))
                    - COUNT({expression}) AS nonnumeric_n
            FROM sparcs_raw
            """
        ).df()
    )

    nonnumeric_results.append(
        con.sql(
            f"""
            SELECT
                {quote_literal(column)} AS column_name,
                {column_sql} AS raw_value,
                COUNT(*) AS value_n
            FROM sparcs_raw
            WHERE NULLIF(TRIM({column_sql}), '') IS NOT NULL
              AND {expression} IS NULL
            GROUP BY {column_sql}
            ORDER BY value_n DESC
            """
        ).df()
    )

numeric_audit = pd.concat(numeric_results, ignore_index=True)

nonnumeric_values = pd.concat(
    nonnumeric_results,
    ignore_index=True,
)

display(numeric_audit)
display(nonnumeric_values)

,column_name,total_n,nonmissing_n,numeric_n,nonnumeric_n
0,Length of Stay,2125754,2125754,2123464,2290
1,Birth Weight,2125754,204003,203882,121
2,Total Charges,2125754,2125754,2125754,0
3,Total Costs,2125754,2125754,2125754,0
4,APR DRG Code,2125754,2125754,2125754,0
5,APR MDC Code,2125754,2125754,2125754,0
6,APR Severity of Illness Code,2125754,2125754,2125754,0


,column_name,raw_value,value_n
0,Length of Stay,120 +,2290
1,Birth Weight,UNKN,121


## 10. LOS distribution and top-coding

In [12]:
los_expression = """
    CASE
        WHEN REGEXP_MATCHES(
            TRIM("Length of Stay"),
            '^120\\s*\\+$'
        )
        THEN 120.0
        ELSE TRY_CAST(TRIM("Length of Stay") AS DOUBLE)
    END
"""

los_distribution = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_n,
        COUNT({los_expression}) AS valid_n,
        MIN({los_expression}) AS minimum,
        AVG({los_expression}) AS mean_lower_bound,
        APPROX_QUANTILE({los_expression}, 0.25) AS p25,
        APPROX_QUANTILE({los_expression}, 0.50) AS median,
        APPROX_QUANTILE({los_expression}, 0.75) AS p75,
        APPROX_QUANTILE({los_expression}, 0.95) AS p95,
        APPROX_QUANTILE({los_expression}, 0.99) AS p99,
        MAX({los_expression}) AS observed_upper_value,
        SUM(
            CASE
                WHEN REGEXP_MATCHES(
                    TRIM("Length of Stay"),
                    '^120\\s*\\+$'
                )
                THEN 1
                ELSE 0
            END
        ) AS top_coded_n
    FROM sparcs_raw
    """
).df()

los_distribution["top_coded_pct"] = (
    los_distribution["top_coded_n"] / total_rows * 100
)

los_distribution

,total_n,valid_n,minimum,mean_lower_bound,p25,median,p75,p95,p99,observed_upper_value,top_coded_n,top_coded_pct
0,2125754,2125754,1.0,5.780398,2.0,3.0,6.058762,18.566067,42.180382,120.0,2290.0,0.107726


### Interpretation

Length of stay is strongly right-skewed. The lower-bound mean is **5.78 days**,
compared with a median of **3 days**. The 95th and 99th percentiles are
approximately **18.53 days** and **42.20 days**, respectively.

There are **2,290 records**, representing approximately **0.108%** of the
dataset, recorded as `120 +`.

After substituting 120 as the observable lower bound, the calculated mean is
a lower-bound estimate rather than the exact mean LOS. These observations must
not automatically be treated as exact 120-day stays in the later modeling
workflow.

## 11. Financial distributions

In [13]:
financial_results = []

for column in ["Total Charges", "Total Costs"]:
    if column not in columns:
        continue

    expression = numeric_expression(column)

    result = con.sql(
        f"""
        SELECT
            {quote_literal(column)} AS metric,
            COUNT(*) AS total_n,
            COUNT({expression}) AS valid_n,
            MIN({expression}) AS minimum,
            AVG({expression}) AS mean,
            APPROX_QUANTILE({expression}, 0.25) AS p25,
            APPROX_QUANTILE({expression}, 0.50) AS median,
            APPROX_QUANTILE({expression}, 0.75) AS p75,
            APPROX_QUANTILE({expression}, 0.95) AS p95,
            APPROX_QUANTILE({expression}, 0.99) AS p99,
            MAX({expression}) AS maximum,
            SUM(
                CASE WHEN {expression} <= 0 THEN 1 ELSE 0 END
            ) AS nonpositive_n
        FROM sparcs_raw
        """
    ).df()

    financial_results.append(result)

financial_distribution = pd.concat(
    financial_results,
    ignore_index=True,
)

financial_distribution

,metric,total_n,valid_n,minimum,mean,p25,median,p75,p95,p99,maximum,nonpositive_n
0,Total Charges,2125754,2125754,1.50,83340.272126,22005.332192,43931.391809,88046.632896,266473.241311,660245.991505,25686720.00,0.0
1,Total Costs,2125754,2125754,0.09,25155.113004,6880.447782,13228.173111,26539.668897,78808.715759,201460.628902,11127210.71,0.0


### Interpretation

Charges and estimated costs are strongly right-skewed.

- `Total Charges` has a mean of approximately **$83,340** and a median of
  approximately **$43,901**. The maximum is approximately **$25.69 million**.
- `Total Costs` has a mean of approximately **$25,155** and a median of
  approximately **$13,234**. The maximum is approximately **$11.13 million**.

Medians and percentiles are therefore required alongside means. Later
modeling should consider log transformations, robust loss functions, or other
methods appropriate for heavily skewed financial outcomes.

`Total Charges` represents billed charges—not revenue, reimbursement, or
operating cost. `Total Costs` is an analytical cost estimate and should not be
described as audited hospital expense.

## 12. Apparent duplicate-pattern assessment

In [14]:
all_columns_sql = ",\n".join(
    quote_identifier(column)
    for column in columns
)

duplicate_summary = con.sql(
    f"""
    WITH duplicate_patterns AS (
        SELECT
            {all_columns_sql},
            COUNT(*) AS pattern_n
        FROM sparcs_raw
        GROUP BY
            {all_columns_sql}
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicated_patterns,
        COALESCE(SUM(pattern_n), 0) AS rows_in_duplicated_patterns,
        COALESCE(SUM(pattern_n - 1), 0) AS excess_rows
    FROM duplicate_patterns
    """
).df()

duplicate_summary

,duplicated_patterns,rows_in_duplicated_patterns,excess_rows
0,2772,6736.0,3964.0


### Interpretation

The audit identified **2,772 fully repeated value patterns** involving
**6,736 rows**. These patterns represent **3,964 excess rows** beyond the
first occurrence of each repeated pattern.

These records were not removed. The public-use dataset does not contain a
unique discharge identifier, and de-identification may cause genuinely
different discharges to have identical released values. The audit can identify
repeated patterns but cannot prove that they are erroneous duplicate records.

## 13. Facility-Key consistency

In [15]:
facility_key_audit = pd.DataFrame()
facility_name_audit = pd.DataFrame()

required_facility_fields = {
    "Permanent Facility Id",
    "Facility Name",
}

if required_facility_fields.issubset(columns):
    facility_key_audit = con.sql(
        """
        SELECT
            "Permanent Facility Id",
            COUNT(
                DISTINCT NULLIF(TRIM("Facility Name"), '')
            ) AS facility_name_count,
            COUNT(*) AS discharges
        FROM sparcs_raw
        WHERE NULLIF(
            TRIM("Permanent Facility Id"),
            ''
        ) IS NOT NULL
        GROUP BY "Permanent Facility Id"
        HAVING COUNT(
            DISTINCT NULLIF(TRIM("Facility Name"), '')
        ) > 1
        ORDER BY facility_name_count DESC, discharges DESC
        """
    ).df()

    facility_name_audit = con.sql(
        """
        SELECT
        TRIM("Facility Name") AS facility_name,
        LIST(
            DISTINCT NULLIF(
                TRIM("Permanent Facility Id"),
                ''
            )
            ORDER BY NULLIF(
                TRIM("Permanent Facility Id"),
                ''
            )
        ) AS facility_ids,
        COUNT(
            DISTINCT NULLIF(
                TRIM("Permanent Facility Id"),
                ''
            )
        ) AS facility_id_count,
        COUNT(*) AS discharges
        FROM sparcs_raw
        WHERE NULLIF(TRIM("Facility Name"), '') IS NOT NULL
        GROUP BY TRIM("Facility Name")
        HAVING COUNT(
            DISTINCT NULLIF(
                TRIM("Permanent Facility Id"),
                ''
            )
        ) > 1
        ORDER BY facility_id_count DESC, discharges DESC
        """
    ).df()

display(facility_key_audit)
display(facility_name_audit)

,Permanent Facility Id,facility_name_count,discharges


,facility_name,facility_ids,facility_id_count,discharges
0,BronxCare Hospital Center,"[001164, 001178]",2,24366
1,The University of Vermont Health Network - Alice Hyde Medical Center,"[000325, 015485]",2,762


### Interpretation

No `Permanent Facility Id` was associated with more than one facility name
under the current normalized-name comparison.

Two facility names were associated with more than one permanent facility ID:

- `BronxCare Hospital Center`, covering 24,366 discharge records
- `The University of Vermont Health Network - Alice Hyde Medical Center`,
  covering 762 discharge records

This does not prove that either mapping is erroneous. A shared name may
represent multiple facilities, campuses, identifier changes, or public-file
standardization.

`Permanent Facility Id` should therefore be used as the primary facility key
in the analytical model. Facility-name-to-ID mappings should be investigated
before constructing the final facility dimension.

## 14. Privacy and de-identification review

In [16]:
privacy_fields = [
    "Age Group",
    "Zip Code - 3 digits",
    "Hospital Service Area",
    "Hospital County",
    "Operating Certificate Number",
    "Permanent Facility Id",
    "Facility Name",
]

privacy_profile = column_profile.loc[
    column_profile["column_name"].isin(privacy_fields)
].copy()

privacy_categories = categorical_counts.loc[
    categorical_counts["column_name"].isin(privacy_fields)
].copy()

display(privacy_profile)
display(privacy_categories.groupby(
    "column_name",
    group_keys=False,
).head(20))

,column_name,missing_n,unique_n,missing_pct
5,Zip Code - 3 digits,41883.0,50,1.970266
6,Permanent Facility Id,5333.0,207,0.250876
7,Operating Certificate Number,5333.0,160,0.250876
8,Hospital County,5333.0,57,0.250876
9,Hospital Service Area,5333.0,8,0.250876
18,Facility Name,0.0,206,0.000000
25,Age Group,0.0,5,0.000000


,column_name,category,discharges,percentage
0,Hospital Service Area,New York City,963525,45.3263
1,Hospital Service Area,Long Island,346694,16.3092
2,Hospital Service Area,Hudson Valley,237546,11.1747
3,Hospital Service Area,Western NY,138654,6.5226
4,Hospital Service Area,Finger Lakes,137867,6.4856
...,...,...,...,...
662,Zip Code - 3 digits,140,37746,1.7757
663,Zip Code - 3 digits,130,34788,1.6365
664,Zip Code - 3 digits,119,32207,1.5151
665,Zip Code - 3 digits,145,29966,1.4097


### Interpretation

The public-use file applies several observable de-identification and
generalization mechanisms:

- Age is released in five broad age groups rather than as exact age.
- ZIP code is limited to three digits and is missing for **41,883 records**
  or approximately **1.97%** of the dataset.
- Hospital service area, hospital county, operating certificate number, and
  permanent facility ID each contain **5,333 missing values**, or approximately
  **0.25%** of the dataset. The current audit has not yet confirmed whether
  these missing values occur on exactly the same rows.
- The file contains no direct patient identifier or unique public discharge
  identifier.

The project will retain fewer-than-11 display suppression as a conservative
dashboard standard. This notebook does not claim that 11 is a universal
SPARCS publication requirement.

## 15. Export Audit Outputs

In [17]:
audit_outputs = {
    "file_metadata.csv": file_metadata,
    "schema.csv": schema,
    "required_field_availability.csv": field_availability,
    "column_profile.csv": column_profile,
    "categorical_counts.csv": categorical_counts,
    "numeric_audit.csv": numeric_audit,
    "nonnumeric_values.csv": nonnumeric_values,
    "los_distribution.csv": los_distribution,
    "financial_distribution.csv": financial_distribution,
    "duplicate_summary.csv": duplicate_summary,
    "facility_key_audit.csv": facility_key_audit,
    "facility_name_audit.csv": facility_name_audit,
}

export_manifest = []

for file_name, dataframe in audit_outputs.items():
    output_path = OUTPUT_DIR / file_name
    dataframe.to_csv(output_path, index=False)

    assert output_path.exists()

    export_manifest.append(
        {
            "file_name": file_name,
            "row_count": len(dataframe),
            "output_path": str(output_path.relative_to(PROJECT_ROOT)
),
        }
    )

export_manifest = pd.DataFrame(export_manifest)
export_manifest

,file_name,row_count,output_path
0,file_metadata.csv,1,outputs\data_audit\file_metadata.csv
1,schema.csv,33,outputs\data_audit\schema.csv
2,required_field_availability.csv,21,outputs\data_audit\required_field_availability.csv
3,column_profile.csv,33,outputs\data_audit\column_profile.csv
4,categorical_counts.csv,779,outputs\data_audit\categorical_counts.csv
5,numeric_audit.csv,7,outputs\data_audit\numeric_audit.csv
6,nonnumeric_values.csv,2,outputs\data_audit\nonnumeric_values.csv
7,los_distribution.csv,1,outputs\data_audit\los_distribution.csv
8,financial_distribution.csv,2,outputs\data_audit\financial_distribution.csv
9,duplicate_summary.csv,1,outputs\data_audit\duplicate_summary.csv


## 16. Notebook 01 summary

### Work Completed

- Validated the 2023 SPARCS source file and schema.
- Documented file metadata and a reproducibility hash.
- Measured missingness and cardinality for all source fields.
- Reviewed categorical values and observable de-identification patterns.
- Validated numeric parsing for LOS, charges, costs, and related fields.
- Examined skewness and LOS top-coding.
- Assessed fully repeated record patterns without deleting records.
- Evaluated facility-key consistency.
- Exported reproducible audit tables.

### Key Findings

1. **Dataset structure:** The file contains 2,125,754 released inpatient
   discharge records and 33 fields. Its intended grain is one discharge record
   per row, but no unique discharge identifier is available to independently
   verify row-level uniqueness.

2. **Missingness:** The largest missingness occurs in birth weight, secondary
   and tertiary payer fields, and CCSR procedure fields. Much of this may be
   structural or not-applicable, but the assumptions require confirmation
   during data cleaning.

3. **Length of stay:** LOS is strongly right-skewed, with a lower-bound mean of
   5.78 days and a median of 3 days. A total of 2,290 records are top-coded as
   `120 +`.

4. **Charges and costs:** Both fields are strongly right-skewed. Mean charges
   and costs are approximately 1.9 times their respective medians, and both
   contain extreme upper-tail observations.

5. **Facility identifiers:** No permanent facility ID maps to multiple
   facility names under the current comparison. Two facility names map to
   multiple permanent facility IDs and require review before construction of
   the facility dimension.

6. **Privacy and suppression:** Age and ZIP code are generalized, some
   geographic and facility identifiers are missing, and the file contains no
   direct patient or unique discharge identifier.

### Modeling Implications

- The primary modeling objective is retrospective case-mix-adjusted LOS
  estimation, not admission-time clinical prediction.
- The primary model will exclude LOS-derived variables, charges, estimated
  costs, patient disposition, and other downstream resource-use information.
- APR-DRG, severity, diagnosis, and procedure fields may be considered because
  the objective is retrospective benchmarking, but their availability and
  interpretation must be documented.
- Hospital identity will be evaluated only through sensitivity analysis because
  including it in the primary model could absorb the operational differences
  the project is intended to measure.
- Top-coded LOS records should retain a top-code indicator and should be
  evaluated using sensitivity or censoring-aware approaches.
- Because only 2023 is available, temporal stability cannot be demonstrated.
  Validation should test generalization across hospitals or appropriate data
  partitions.

### Power BI Implications

- Supported time granularity is annual only.
- Higher-cardinality categorical fields include facility, CCSR diagnosis,
  CCSR procedure, and APR-DRG.
- Continuous charges and costs should be modeled as measures rather than
  categorical dimensions.
- Birth weight should normally be excluded from the general inpatient semantic
  model unless a neonatal-specific analysis is developed.
- Secondary and tertiary payer fields should be evaluated before inclusion
  because of substantial missingness.
- Required dimensions include facility, geography, demographics, admission
  type, patient disposition, APR-DRG and severity, diagnosis, procedure,
  payer, emergency department indicator, and discharge year.
- Monthly trends, seasonality, patient-level utilization, readmissions, and
  longitudinal member analysis are not supported by this single-year,
  discharge-level public file.

### Important Limitations

- The dataset contains no unique public discharge identifier.
- Repeated rows cannot automatically be classified as erroneous duplicates.
- Top-coded LOS values do not reveal the exact stay duration.
- Charges do not represent reimbursement or revenue.
- Estimated costs do not represent audited operating expenses.
- Multi-year schema compatibility has not yet been established.